# Exploratory Data Analysis — OECD Inter-Country Input-Output Matrix (2022 Release)

This notebook performs a systematic examination of the OECD Inter-Country Input-Output (ICIO) table for the reference year 2022. The ICIO framework decomposes gross output across countries and industries into intermediate deliveries, value-added payments, and final demand components, permitting a rigorous mapping of productive interdependencies at the global level. The analysis proceeds in three stages.
1) The raw matrix is loaded and its structural properties are inspected; the final-demand suffixes are identified and the overall dimensions of the dataset are confirmed against the OECD's published documentation.
2) The country and sector codes are extracted from both the row and column labels and verified against the expected taxonomy of 81 countries and 50 sectors; the three special accounting rows (`VA`, `TLS`, `OUT`) are identified and separated from the country-sector units.
3) The intermediate-transactions block $Z$ is isolated by filtering both rows and columns to retain only entries that conform to the `COUNTRY_SECTOR` convention, yielding the square sub-matrix of dimension $4{,}250 \times 4{,}250$ that serves as the input to all subsequent matrix operations in the downstream notebooks.


## 1. Data and Libraries loading

The analysis starts by importing the necessary libraries and loading the OECD ICIO data for 2022.
Then, according to the OECD's documentation, the final demand suffixes are selected, as they will be required for the subsequent analysis of final demand components. The six categories are:
- Household final consumption expenditure (`HFCE`)
- Expenditure by non-profit institutions serving households (`NPISH`)
- General government final consumption (`GGFC`)
- Gross fixed capital formation (`GFCF`)
- Changes in inventories (`INVNT`)
- Direct purchases abroad by residents (`DPABR`)

Together they partition all final (non-intermediate) uses of domestic and imported output, so their correct identification is a prerequisite for separating the intermediate-transactions block from the final-demand block in subsequent cells.

Finally, the data is imported and the dimensions of the resulting data frame are checked to ensure that the expected number of rows and columns are present, confirming that the dataset has been loaded correctly.

In [32]:
# Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import re

# Set path constants
DATA_RAW  = Path("../data/raw")
DATA_PREP = Path("../data/prepared")
FIG_DIR   = Path("../outputs/figures")
TBL_DIR   = Path("../outputs/tables")

# Load data with row and column labels as index and low_memory option to avoid dtype warnings
raw_icio_table = pd.read_csv(DATA_RAW / "2022.csv", low_memory=False, index_col=0)
print("ICIO table imported succesfully")

# Display dimensions and first few rows and columns of the data frame for a final check
print(f"The table has {raw_icio_table.shape[0]} rows and {raw_icio_table.shape[1]} columns")
print("-"*50)
print("Displaying first 5 rows and 5 columns:")
print(raw_icio_table.iloc[:5, :5])

ICIO table imported succesfully
The table has 4253 rows and 4737 columns
--------------------------------------------------
Displaying first 5 rows and 5 columns:
           AGO_A01   AGO_A02   AGO_A03  AGO_B05  AGO_B06
V1                                                      
AGO_A01  3617.6076   29.7075   76.6695   0.0294   5.8791
AGO_A02    43.8075  171.8510    2.5505   0.1942   6.2443
AGO_A03     4.5608    0.4682  436.9011   0.0072   0.0938
AGO_B05     0.0845    0.0035    0.0112   0.0383   1.1811
AGO_B06     0.0720    0.0009    0.0063   0.0872  94.1575


The OECD ICIO table is distributed as a single wide CSV file in which each row represents a producing country-sector combination and each column represents a using country-sector combination or a final-demand category. The first column carries row labels of the form `COUNTRY_SECTOR` (`AGO_A01`).

## 2. Countries and Sectors Identification
According to the OECD's documentation, the ICIO tabble includes 50 sectors and 81 countries (where `ROW` stands for rest of the world), which implies that the intermediate-transactions block should consist of 4,050 columns (81 countries x 50 sectors). The remaining columns correspond to the six final-demand categories and to the three special items: `VA` (value added), `TLS` (taxes less subsidies on products), and `OUT` (gross output). These rows do not correspond to any specific country-sector combination and should be excluded from the counts of countries and sectors.
The cell below extracts the country and sector codes from the row labels, counts the unique entries, and verifies that they match the expected numbers. It also prints the dimensions of the data frame for a final check.

In [26]:
# With regex, retrieve the unique countries prefixes in the dataset
country_codes = sorted(set(c.split("_", 1)[0] for c in raw_icio_table.columns.to_list()))
# Country codes according to the OECD's documentation
expected_country_codes = [
    'AGO', 'JOR', 'ARE', 'JPN', 'ARG', 'KAZ', 'AUS', 'KHM', 'AUT', 'KOR',
    'BEL', 'LAO', 'BGD', 'LTU', 'BGR', 'LUX', 'BLR', 'LVA', 'BRA', 'MAR',
    'BRN', 'MEX', 'CAN', 'MLT', 'CHE', 'MMR', 'CHL', 'MYS', 'CHN', 'NGA',
    'CIV', 'NLD', 'CMR', 'NOR', 'COD', 'NZL', 'COL', 'PAK', 'CRI', 'PER',
    'CYP', 'PHL', 'CZE', 'POL', 'DEU', 'PRT', 'DNK', 'ROU', 'EGY', 'RUS',
    'ESP', 'SAU', 'EST', 'SEN', 'FIN', 'SGP', 'FRA', 'STP', 'GBR', 'SVK',
    'GRC', 'SVN', 'HKG', 'SWE', 'HRV', 'THA', 'HUN', 'TUN', 'IDN', 'TUR',
    'IND', 'TWN', 'IRL', 'UKR', 'ISL', 'USA', 'ISR', 'VNM', 'ITA', 'ZAF',
    'ROW'
]
# Check that the retrieved country codes match the expected ones
if set(country_codes) == set(expected_country_codes):
    print("All country codes match the expected ones.")
else:
    mismatch_countries = set(country_codes).symmetric_difference(set(expected_country_codes))
    print(f"Unique country codes found in columns : {len(country_codes)}")
    print(country_codes)
    print("-"*50)
    print(f"Mismatched country codes: {mismatch_countries}")


# With regex, retrieve the unique sector suffixes in the dataset
sector_codes = sorted(set(c.split("_", 1)[1] for c in raw_icio_table.columns.to_list() if "_" in c))
# Final-demand column suffixes used by OECD ICIO 2022 release
final_demand_suffixes = ("HFCE", "NPISH", "GGFC", "GFCF", "INVNT", "DPABR")
# Remove final-demand suffixes from the set of sector codes
sector_codes = [s for s in sector_codes if s not in final_demand_suffixes]
print(f"Unique sector codes found in columns : {len(sector_codes)}")
print(sector_codes)

Unique country codes found in columns : 86
['AGO', 'ARE', 'ARG', 'AUS', 'AUT', 'BEL', 'BGD', 'BGR', 'BLR', 'BRA', 'BRN', 'CAN', 'CHE', 'CHL', 'CHN', 'CIV', 'CMR', 'CN1', 'CN2', 'COD', 'COL', 'CRI', 'CYP', 'CZE', 'DEU', 'DNK', 'EGY', 'ESP', 'EST', 'FIN', 'FRA', 'GBR', 'GRC', 'HKG', 'HRV', 'HUN', 'IDN', 'IND', 'IRL', 'ISL', 'ISR', 'ITA', 'JOR', 'JPN', 'KAZ', 'KHM', 'KOR', 'LAO', 'LTU', 'LUX', 'LVA', 'MAR', 'MEX', 'MLT', 'MMR', 'MX1', 'MX2', 'MYS', 'NGA', 'NLD', 'NOR', 'NZL', 'OUT', 'PAK', 'PER', 'PHL', 'POL', 'PRT', 'ROU', 'ROW', 'RUS', 'SAU', 'SEN', 'SGP', 'STP', 'SVK', 'SVN', 'SWE', 'THA', 'TUN', 'TUR', 'TWN', 'UKR', 'USA', 'VNM', 'ZAF']
--------------------------------------------------
Mismatched country codes: {'CN1', 'MX1', 'CN2', 'MX2', 'OUT'}
Unique sector codes found in columns : 50
['A01', 'A02', 'A03', 'B05', 'B06', 'B07', 'B08', 'B09', 'C10T12', 'C13T15', 'C16', 'C17_18', 'C19', 'C20', 'C21', 'C22', 'C23', 'C24A', 'C24B', 'C25', 'C26', 'C27', 'C28', 'C29', 'C301', 'C302T309',

By identifying the distinct countries and sectors codes, it is possible to confirm that the dataset includes the expected 81 countries and 50 sectors, which is crucial for ensuring the integrity of the subsequent analysis. However, there are two minor discrepancies:
- The column labels include also the special item `OUT` (gross output), which does not correspond to any specific country-sector combination and should be excluded from the counts of countries and sectors. However, this was specified in the OECD's documentation and does not affect the overall structure of the dataset, as it is a common practice to include such aggregate rows for reference.
- The column labels include strange entries such as `CN1` and `CN2`, which do not correspond to any known country code and should be investigated further to determine their origin and whether they should be included or excluded from the analysis. These entries could potentially represent distinction between different types of Chinese output (e.g., domestic vs. processing), but this is not clarified in the documentation and warrants further investigation before proceeding with the analysis.

It is possible to replicate the same procedure for the row labels, which should yield the same number of countries and sectors, confirming that the dataset is internally consistent and correctly structured for subsequent analysis. According to the OECD's documentation, the rows should also include three special items: `VA` (value added), `TLS` (taxes less subsidies on products), and `OUT` (gross output), which do not correspond to any specific country-sector combination and should be excluded from the counts of countries and sectors. 

In [ ]:
# Retrieve the unique sector codes in the row labels, which should be the same as those in the column labels
row_sector_codes = sorted(set(r.split("_", 1)[1] for r in raw_icio_table.index.to_list() if "_" in r))
print(f"Unique sector codes found in rows : {len(row_sector_codes)}")
print(row_sector_codes)

# Retrieve the unique country prefixes in the row labels, which should be the same as those in the column labels
row_country_codes = sorted(set(r.split("_", 1)[0] for r in raw_icio_table.index.to_list()))
print(f"Unique country codes found in rows : {len(row_country_codes)}")
print(row_country_codes)
if set(row_country_codes) == set(expected_country_codes):
    print("All country codes in rows match the expected ones.")
else:
    mismatch_countries = set(row_country_codes).symmetric_difference(set(expected_country_codes))
    print(f"Mismatched country codes in rows: {mismatch_countries}")


Unique sector codes found in rows : 50
['A01', 'A02', 'A03', 'B05', 'B06', 'B07', 'B08', 'B09', 'C10T12', 'C13T15', 'C16', 'C17_18', 'C19', 'C20', 'C21', 'C22', 'C23', 'C24A', 'C24B', 'C25', 'C26', 'C27', 'C28', 'C29', 'C301', 'C302T309', 'C31T33', 'D', 'E', 'F', 'G', 'H49', 'H50', 'H51', 'H52', 'H53', 'I', 'J58T60', 'J61', 'J62_63', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T']
Unique country codes found in rows : 88
['AGO', 'ARE', 'ARG', 'AUS', 'AUT', 'BEL', 'BGD', 'BGR', 'BLR', 'BRA', 'BRN', 'CAN', 'CHE', 'CHL', 'CHN', 'CIV', 'CMR', 'CN1', 'CN2', 'COD', 'COL', 'CRI', 'CYP', 'CZE', 'DEU', 'DNK', 'EGY', 'ESP', 'EST', 'FIN', 'FRA', 'GBR', 'GRC', 'HKG', 'HRV', 'HUN', 'IDN', 'IND', 'IRL', 'ISL', 'ISR', 'ITA', 'JOR', 'JPN', 'KAZ', 'KHM', 'KOR', 'LAO', 'LTU', 'LUX', 'LVA', 'MAR', 'MEX', 'MLT', 'MMR', 'MX1', 'MX2', 'MYS', 'NGA', 'NLD', 'NOR', 'NZL', 'OUT', 'PAK', 'PER', 'PHL', 'POL', 'PRT', 'ROU', 'ROW', 'RUS', 'SAU', 'SEN', 'SGP', 'STP', 'SVK', 'SVN', 'SWE', 'THA', 'TLS', 'TUN', 'TUR',

Hence, the row index of the raw ICIO table contains two structurally distinct types of labels. 
- The large majority follow the `COUNTRY_SECTOR` convention
- `VA` (value added), `TLS` (taxes less subsidies on products), and `OUT` (gross output) are the accounting aggregates that the OECD appends at the bottom of the transaction block. They record, for each country-sector column, the primary factor payments, net product taxes, and total gross output respectively. Retaining them in the main data frame would introduce phantom producing units and corrupt all subsequent matrix operations.

## 3. ICIO Table Structure - Z-block isolation

According to the OECD's documentation, the ICIO table is structured as follows: the first 4,250 columns (85 countries x 50 sectors) represent the intermediate transactions between country-sector units, while the remaining columns correspond to final demand categories and special items. The rows are similarly structured, with the majority following the `COUNTRY_SECTOR` convention and three special rows for value added, taxes less subsidies on products, and gross output.

The Z block is the square submatrix of the ICIO table that captures the intermediate transactions between country-sector units. Isolating this block is a crucial step for subsequent analyses, as it allows us to focus on the interdependencies between producing units without the confounding influence of final demand components and aggregate rows. 

In [40]:
# Convert to set for faster lookup
sector_code_set = set(sector_codes)  

# Filter columns: keep only those that follow the `COUNTRY_SECTOR` convention and whose sector code is in the identified set
intermediate_columns = [c for c in raw_icio_table.columns if "_" in c and c.split("_", 1)[1] in sector_code_set]

# Filter rows: same logic
intermediate_rows = [r for r in raw_icio_table.index if "_" in r and r.split("_", 1)[1] in sector_code_set]

# Display the number of identified intermediate transaction columns and rows to confirm that they match the expected dimensions
print(f"Number of intermediate transaction columns: {len(intermediate_columns)}")
print(f"Number of intermediate transaction rows: {len(intermediate_rows)}")

# Isolate the Z-block of intermediate transactions using the identified row and column labels
z_block = raw_icio_table.loc[intermediate_rows, intermediate_columns]
print("Z-block of intermediate transactions isolated successfully.")
print(f"Z-block dimensions: {z_block.shape}")


Number of intermediate transaction columns: 4250
Number of intermediate transaction rows: 4250
Z-block of intermediate transactions isolated successfully.
Z-block dimensions: (4250, 4250)


The result matches the expected dimensions of 4,250 rows and 4,250 columns, confirming that the Z block has been correctly isolated from the full ICIO table.